# Combine annotation CSV outputs

Input CSVs contain crowd-sourced labels for items in the Ukrainian folk art collection. Each row has an object id (`europeana_id`), a label string (`value`), and vote counts. The same object–label pair can appear in more than one export if it was collected in more than one event.

This notebook merges on `(europeana_id, value)`, sums `upvotes` and `downvotes` across files, and sets `recommendation` from those totals: `accept`, `reject`, `tie`, or `unknown`.

## Setup

Define a common `BASE_DIR` for file paths and, when running in Google Colab, optionally mount Google Drive.

In [32]:
from pathlib import Path

IN_COLAB = False
try:
  from google.colab import drive  # type: ignore
  IN_COLAB = True
except ImportError:
  pass

if IN_COLAB:
  drive.mount('/content/drive')

USE_DRIVE = IN_COLAB

if USE_DRIVE:
  BASE_DIR = Path('/content/drive/MyDrive/AISTER-Crowdsourcing-Pilot/step_3')
else:
  BASE_DIR = Path('../')

In [ ]:
print(f'Base directory: {BASE_DIR.resolve()}')

## Inputs and output

Set `INPUT_CSVS` to every export you want merged; order does not matter.

In [34]:
import pandas as pd

OUTPUTS_DIR = BASE_DIR / 'outputs'
INPUT_CSVS = [
  OUTPUTS_DIR / '1_ukrainian-folkart-annotations.csv',
  OUTPUTS_DIR / '2_ukrainian-folkart-annotations.csv',
  OUTPUTS_DIR / '3_ukrainian-folkart-annotations.csv',
  OUTPUTS_DIR / '4_ukrainian-folkart-annotations.csv',
]
OUTPUT_COMBINED = OUTPUTS_DIR / 'combined_ukrainian-folkart-annotations.csv'

## Load and merge

Load every batch CSV into one table.

In [35]:
frames = []
for path in INPUT_CSVS:
  if not path.exists():
    raise FileNotFoundError(path)
  frames.append(pd.read_csv(path))

all_rows = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(all_rows)} rows from {len(INPUT_CSVS)} files')

Loaded 18625 rows from 4 files


The same tag on the same object may appear in multiple event files. Rows are grouped and `upvotes` / `downvotes` are summed.

In [36]:
g = all_rows.groupby(['europeana_id', 'value'], as_index=False)
merged = g.agg(
  upvotes=('upvotes', 'sum'),
  downvotes=('downvotes', 'sum'),
)

`recommendation` follows only the summed votes, not the labels in the source CSVs.

In [37]:
def recommendation_from_votes(up: int, down: int) -> str:
  if up > down:
    return 'accept'
  if down > up:
    return 'reject'
  if up == down and up > 0:
    return 'tie'
  return 'unknown'


merged['recommendation'] = [
  recommendation_from_votes(int(u), int(d))
  for u, d in zip(merged['upvotes'], merged['downvotes'])
]

### Column order and preview

Keep the export schema, sort for stable diffs, then show the first rows.

In [38]:
merged = merged[
  ['europeana_id', 'value', 'upvotes', 'downvotes', 'recommendation']
]
merged = merged.sort_values(by=['europeana_id', 'value']).reset_index(drop=True)

In [39]:
# Print some random rows
merged.sample(20)

,europeana_id,value,upvotes,downvotes,recommendation
5350,KYD1916,depicting a scene,6,1,accept
3357,KYD1807,red umbrella,13,2,accept
4457,KYD1874,blue roof,16,0,accept
737,KYD1252,wooden chair,24,0,accept
3898,KYD1844,leaves,17,0,accept
2591,KYD1771,Windmills,13,0,accept
5200,KYD1908,girl,4,9,reject
4511,KYD1877,Traditional clothing,13,0,accept
5828,KYD495,wooden fence,2,0,accept
4550,KYD1878,spruce tree,2,0,accept


## Stats

Counts of rows by final `recommendation`.

In [40]:
stats = merged['recommendation'].value_counts()
for label in ('accept', 'reject', 'tie', 'unknown'):
  print(f'{label}: {stats.get(label, 0)}')

accept: 5425
reject: 370
tie: 24
unknown: 10


## Export

In [41]:
merged.to_csv(OUTPUT_COMBINED, index=False)
print(f'Wrote: {OUTPUT_COMBINED}')

Wrote: ../outputs/combined_ukrainian-folkart-annotations.csv
